## SVMs

### Homework: Implement a Linear SVM (Classification)

In this assignment, you will implement a **linear Support Vector Machine (SVM)** for binary classification.  
You are given the class skeleton below; your task is to complete the `fit` and `predict` methods.  
Assume labels are binary and convert them internally to \(\{-1, +1\}\). The goal is to correctly learn the weight vector \(w\) and bias \(b\) and use them to make predictions.


In [ ]:
import numpy as np

class SVM:

    def __init__(self, learning_rate=0.001, lambda_param=0.01, n_iters=1000):
        self.lr = learning_rate
        self.lambda_param = lambda_param
        self.n_iters = n_iters
        self.w = None
        self.b = None

    def fit(self, X, y):
        y_ = np.where(y <= 0, -1, 1)
        n_samples, n_features = X.shape

        self.w = np.zeros(n_features)
        self.b = 0

        for _ in range(self.n_iters):
            for idx, x_i in enumerate(X):
                condition = y_[idx] * (np.dot(x_i, self.w) + self.b) >= 1

                if condition:
                    self.w -= self.lr * (2 * self.lambda_param * self.w)
                else:
                    self.w -= self.lr * (2 * self.lambda_param * self.w - np.dot(x_i, y_[idx]))
                    self.b -= self.lr * (-y_[idx])

    def predict(self, X):
        linear_output = np.dot(X, self.w) + self.b
        y_pred = np.where(linear_output >= 0, 1, -1)
        return y_pred

## Decision Trees

### Homework: Implement a Decision Tree Classifier

In this assignment, you will implement a **basic decision tree classifier from scratch**.  
The tree should use the **Gini impurity** to choose splits and should grow recursively.

Stop splitting when a maximum depth is reached **or** when a node contains too few samples.  
You do **not** need to implement preprocessing, feature encoding, or pruning.


In [ ]:
import numpy as np

class DecisionTreeClassifier:
    def __init__(self, max_depth=3, min_size=5):
        self.max_depth = max_depth
        self.min_size = min_size
        self.tree = None

    def gini(self, groups, classes):
        """Compute Gini impurity for a split."""
        n_instances = float(sum([len(group) for group in groups]))

        gini = 0.0
        for group in groups:
            size = float(len(group))
            if size == 0:
                continue

            score = 0.0
            for class_val in classes:
                p = (group[:, -1] == class_val).sum() / size
                score += p * p

            gini += (1.0 - score) * (size / n_instances)

        return gini

    def split_dataset(self, index, value, dataset):
        """Split dataset based on feature index and threshold value."""

        left = dataset[dataset[:, index] < value]
        right = dataset[dataset[:, index] >= value]
        return left, right

    def best_split(self, dataset):
        """Find the best split for a dataset."""
        class_values = np.unique(dataset[:, -1])
        b_index, b_value, b_score, b_groups = 999, 999, 999, None


        for index in range(dataset.shape[1] - 1):
            for row in dataset:
                groups = self.split_dataset(index, row[index], dataset)
                gini = self.gini(groups, class_values)

                if gini < b_score:
                    b_index, b_value, b_score, b_groups = index, row[index], gini, groups

        return {'index': b_index, 'value': b_value, 'groups': b_groups}

    def to_terminal(self, group):
        """Create a terminal node (most common class)."""
        outcomes = group[:, -1]
        unique, counts = np.unique(outcomes, return_counts=True)
        return unique[np.argmax(counts)]

    def build_tree(self, node, depth):
        """Recursively build the decision tree."""
        left, right = node['groups']
        del(node['groups'])

        if len(left) == 0 or len(right) == 0:
            node['left'] = node['right'] = self.to_terminal(np.vstack((left, right)))
            return

        if depth >= self.max_depth:
            node['left'], node['right'] = self.to_terminal(left), self.to_terminal(right)
            return

        if len(left) <= self.min_size:
            node['left'] = self.to_terminal(left)
        else:
            node['left'] = self.best_split(left)
            self.build_tree(node['left'], depth + 1)

        if len(right) <= self.min_size:
            node['right'] = self.to_terminal(right)
        else:
            node['right'] = self.best_split(right)
            self.build_tree(node['right'], depth + 1)

    def fit(self, X, y):
        """Build the decision tree."""
        dataset = np.column_stack((X, y))
        root = self.best_split(dataset)
        self.build_tree(root, depth=1)
        self.tree = root

    def predict_one(self, node, x):
        """Predict class for a single sample."""
        if x[node['index']] < node['value']:

            if isinstance(node['left'], dict):
                return self.predict_one(node['left'], x)
            else:
                return node['left']
        else:

            if isinstance(node['right'], dict):
                return self.predict_one(node['right'], x)
            else:
                return node['right']
    def predict(self, X):
        """Predict classes for multiple samples."""
        return np.array([self.predict_one(self.tree, x) for x in X])